<center>
<img src="https://raw.githubusercontent.com/dvgodoy/PyTorch101_ODSC_Europe2020/master/images/linear_dogs.jpg" width="800">

</center>

# `Домашнее задание 3`

#### Фамилия, имя: `ВАШЕ ИМЯ`

Дата выдачи: <span style="color:red">__21 апреля__</span>.

Дедлайн: <span style="color:red">__28 апреля 23:59__</span>.

Стоимость: __3 балла__ (основная часть заданий) + __2.2 балла__ (бонус) — отнормируем в 10 бальную систему при выставлении в Anytask.

Политика поздних сдач: такая же как на МО-1

<span style="color:red">__В ноутбуке все клетки должны выполняться без ошибок при последовательном их выполнении.__</span>

## Оценивание и штрафы

Задание выполняется самостоятельно. «Похожие» решения считаются плагиатом и все задействованные студенты (в том числе те, у кого списали) не могут получить за него больше 0 баллов. Если вы нашли решение какого-то из заданий (или его часть) в открытом источнике, необходимо указать ссылку на этот источник (скорее всего вы будете не единственным, кто это нашел, поэтому чтобы исключить подозрение в плагиате, необходима ссылка на источник).

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from tqdm.notebook import tqdm

In [ ]:
import torch
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda', index=0)

# Описание данных

Данные можно [скачать с гугл-диска.](https://drive.google.com/drive/folders/11oCcLplWtp_qm-WuEbfCFP_Mz5K_z3ps?usp=sharing) Если вы делаете задание в колабе, то строчки ниже сами скачают вам данные.



In [ ]:
import gdown

url = "https://drive.google.com/drive/folders/11oCcLplWtp_qm-WuEbfCFP_Mz5K_z3ps?usp=sharing"
gdown.download_folder(url, quiet=True, use_cookies=False)

['/content/news_data/ria_news.tsv',
 '/content/news_data/vk_comments.tsv',
 '/content/news_data/vk_news.tsv']

В таблице `ria_news.tsv`  лежат данные о новостях, вышедших на сайте РИА-НОВОСТИ с 15 марта 2018 года по 31 декабря 2018 года.

- `href` - уникальный идентификатор новости (ссылка на неё)
- `date` - дата публикации новости
- `time` - время публикации новости
- `title` - заголовок новости
- `snippet` - краткое описание новости
- `text` - текст новости
- `category` - категория новости
- `keywords` - ключевые слова (подкатегории новости)
- `shows` - счётчик с числом просмотров новости на сайте (на момент парсинга)

In [ ]:
df_ria = pd.read_csv('news_data/ria_news.tsv', sep='\t')
df_ria = df_ria[~df_ria.tags.isnull()]
print(df_ria.shape)
df_ria.head()

(201708, 9)


,href,date,time,title,snippet,text,category,tags,shows
0,/20181231/1548961410.html,2018-12-31,"31 декабря 2018, 23:52",Нетаньяху не собирается в отставку в случае пр...,Премьер-министр Израиля Биньямин Нетаньяху не ...,"МОСКВА, 31 дек - РИА Новости. Премьер-министр ...",В мире,"Биньямин Нетаньяху, Израиль, В мире",728.0
1,/20181231/1548961364.html,2018-12-31,"31 декабря 2018, 23:19",Макрон в новогоднем обращении затронул тему ре...,"Результат реформ не может быть мгновенным, зая...","ПАРИЖ, 31 дек – РИА Новости. Результат реформ ...",В мире,"Эммануэль Макрон, Франция, В мире",3086.0
2,/20181231/1548961337.html,2018-12-31,"31 декабря 2018, 23:12",Аарон Рэмзи проведет переговоры с пятью топ-кл...,"Полузащитник лондонского ""Арсенала"" Аарон Рэмз...","МОСКВА, 31 дек - РИА Новости. Полузащитник лон...",NaN,ФК Арсенал (Лондон),183.0
3,/20181231/1548961304.html,2018-12-31,"31 декабря 2018, 23:09",Гол Азмуна принес сборной Ирана победу над кат...,Футболисты сборной Ирана одержали победу над к...,"МОСКВА, 31 дек - РИА Новости. Футболисты сборн...",NaN,"Сердар Азмун, Сборная Ирана по футболу",78.0
4,/20181231/1548961265.html,2018-12-31,"31 декабря 2018, 23:07",Пятая ракетка мира дель Потро пропустит Открыт...,Аргентинский теннисист Хуан Мартин дель Потро ...,"МОСКВА, 31 дек - РИА Новости. Аргентинский тен...",NaN,Теннис,79.0


Многие новостные агенства поддерживают странички в социальных сетях. Они постят туда самые сочные сюжеты. В таблице `vk_news.tsv` лежат данные о новостях, которые РИА запостили ВКонтакте в период времени с  `2017-09-29 01:28:55` по `2019-02-01 23:13:17`.

- `id` - уникальный идентификатор поста
- `href` - ссылка на сайт (если она была указана в посте)
- `datetime` - дата и время публикации новости
- `title` - заголовок новости
- `text` - текст новости в социальной сети
- `likes` - число лайков под постом
- `comments` - число комментариев под постом

In [ ]:
df_vk = pd.read_csv('news_data/vk_news.tsv', sep='\t')
df_vk['snippet'] = df_vk['text']
df_vk.drop('text', axis=1, inplace=True)
print(df_vk.shape)
df_vk.head()

(19928, 7)


,id,href,datetime,title,likes,comments,snippet
0,24006362,/20190201/1550280358.html,2019-02-01 23:13:17,"В ДНР заявили о задержании диверсантов, причас...",15,28,NaN
1,24006240,/20190201/1550268781.html,2019-02-01 22:38:41,"Житель Урала ""заминировал"" ТЦ из-за снятия со...",32,42,NaN
2,24006100,/20190201/1550282212.html,2019-02-01 21:58:52,"В Черном море нашли ""потерянный флот Гитлера""",84,23,NaN
3,24005972,/20190202/1550283179.html,2019-02-01 21:27:06,В США освободили задержанную российскую актрис...,58,35,NaN
4,24005764,/20190201/1550262848.html,2019-02-01 20:55:54,Толкнувший Скабееву депутат Рады заявил о гроз...,45,145,NaN


В таблице `vk_comments.tsv` лежат комментарии к новостям.

- `id` - уникальный идентификатор комментария
- `post_id` - идентификатор новости, под которой был оставлен комментарий
- `datetime` - дата и время, когда был оставлен комментарий
- `text` - текст комментария
- `likes` - число лайков под комментарием

In [ ]:
df_comments = pd.read_csv('news_data/vk_comments.tsv', sep='\t')
df_comments = df_comments[~df_comments.text.isnull()]
print(df_comments.shape)
df_comments.head()

<ipython-input-8-d189c737f6c0>:1: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df_comments = pd.read_csv('news_data/vk_comments.tsv', sep='\t')


(2612629, 5)


,id,post_id,datetime,text,likes
0,24006366.0,24006362.0,2019-02-01 23:14:14,ЧВК Вагнера?,5.0
1,24006370.0,24006362.0,2019-02-01 23:15:23,"[id4710641|Евгений], выздоравливай.",3.0
2,24006371.0,24006362.0,2019-02-01 23:16:21,"[id442655034|Андрей], искренне желаю этого все...",4.0
3,24006374.0,24006362.0,2019-02-01 23:16:38,Опять про Украину новости?,1.0
4,24006375.0,24006362.0,2019-02-01 23:16:40,Че такое ДНР?,2.0


# А что надо сделать то?

В тетрадке вам предстоит сделать следующие шаги:

1. Вы обучите нейросеть предсказывать категорию новости
2. Вы построите предсказания для тех новостей, где мы ничего не знаем о категории
3. Вы используете уже обученный для сентимент-анализа классификатор из библиотеки `hugging face` чтобы предсказать эмоциональную окраску каждого комментария
4. Вы проведёте аналитику по новостям, а именно построите топы из самых позитивных и негативных категорий и новостей

Для первого шага вам будет дан бэйзлайн. Если вы его прогоните, у вас получится базовая модель, которая даст некоторое качество решения задачи. Вам надо будет выяснить, насколько это качество оказалось хорошим, а затем внести в код некоторые улучшения.



## Часть 1: категоризация новостей (1.2 + 2 бонусных балла)

Каждой новости в соотвествие поставлены ключевые слова. Будем считать, что эти ключевые слова — тематики новости. Нужно научиться предсказывать тематики по тексту новости. Готовые тематики у нас есть только по новостям с сайта. Они за 2018 год. По новостям из ВКонтакте у нас тематик нет. Мы собираемся их предсказать.

Новости, опубликованные ВКонтакте, отличаются от новостей с сайта тем, что у них есть только титул и короткое описание. Странно будет обучать нейросеть на длинных текстах, а потом использовать её на коротких описаниях. Мы не будем так делать. Мы попробуем обучить базовый вариант нейронной сети только на заголовках новостей. Все, кто захочет получить бонусные баллы, смогут попробовать добавить в нейросеть сниппеты (так назыают короткие описания новостей).

## 1.1 Подготовка таргета

Поработаем с таргетом. Мы будем предсказывать переменную `tags`. Давайте выясним скоько уникальных тегов существует.

In [ ]:
from collections import Counter

# удалим все лишние пробелы и сделаем lowercase
df_ria['tags'] = (
    df_ria.tags.
    apply(lambda w: ','.join([item.strip() for item in  w.lower().split(',')]))
)

tags = ','.join(list(df_ria.tags.values))
tags_cnt = Counter(tags.split(','))

print(len(tags_cnt))
tags_cnt.most_common()[-20:]

13344


[('блог анны завершинской об автоспорте - блоги', 1),
 ('министерство транспорта рб', 1),
 ('министерство здравоохранения грузии', 1),
 ('палех', 1),
 ('юрий посохов (хореограф)', 1),
 ('мария александрова', 1),
 ('том бенсон', 1),
 ('абдул каюм кочай', 1),
 ('нуман куртулмуш', 1),
 ('mipim', 1),
 ('владимир попов', 1),
 ('брюно женезио', 1),
 ('роберт фицо', 1),
 ('сергей пашинский', 1),
 ('валерия гонтарева', 1),
 ('нововоронеж', 1),
 ('императорское православное палестинское общество', 1),
 ('event_poslanie_prezidenta_rf_federalnomu_sobraniju', 1),
 ('фхтр', 1),
 ('игорь честин', 1)]

Всего в выборке есть порядка 13 000 тэгов. Многие встречаются всего по разу. Давайте оставим в выборке только те тэги, которые встречаются более 30 раз.

In [ ]:
target_tags = {tag for tag,cnt in tags_cnt.most_common() if cnt > 30}
len(target_tags)

1583

Закодируем теги для OHE.

In [ ]:
tag2idx = dict(zip(target_tags, range(len(target_tags))))
idx2tag = {jtem: item for item,jtem in tag2idx.items()}

Почистим таргет от лишних тэгов.

In [ ]:
df_ria['target_tags'] = (
    df_ria.tags.
    apply(lambda w: [tag2idx.get(item) for item in  w.split(',') if item in target_tags])
)

df_ria = df_ria[df_ria.target_tags.apply(len) > 0]
df_ria.shape

(201437, 10)

In [ ]:
df_ria.target_tags.values[:3]

array([list([422, 261, 937]), list([1424, 1039, 937]), list([1161])],
      dtype=object)

## 1.2 Подготовка текстов

Теперь займёмся предобработкой текстов. Приведём все слова к маленькому регистру и выбросим мусорные символы. В качестве токенов будем рассматривать отдельные слова.

Напомню, что мы пока что решили работать только с названиями статей. Поэтому вся предобработка применяется исключительно к ним. **Спойлер:** предобработку для сниппетов вы сделаете сами в первом же задании.

In [ ]:
import nltk
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [ ]:
import re
from nltk.tokenize import word_tokenize

def normalise_text(text):
    text = text.lower()

    # сурово регулярками выкидываем мусорные символы
    text = re.sub('[^а-яa-z0-9 ]', '', text)
    return text.strip()

df_ria['title_clean'] = df_ria.title.apply(normalise_text)

word_cnt = Counter(word_tokenize(' '.join(df_ria.title_clean.values)))
len(word_cnt)

112178

In [ ]:
word_cnt.most_common()[:10]

[('в', 127323),
 ('на', 44386),
 ('с', 26150),
 ('и', 21771),
 ('о', 19948),
 ('по', 17014),
 ('россии', 13494),
 ('не', 13483),
 ('сша', 9942),
 ('за', 9881)]

Давайте почистим словарь от стоп-слов и подготовим его к использованию внутри датасета. Мы будем с помощью словаря заменять слова на индексы. Добавим в словарь несколько специальных токенов для неизвестных слов и паддингов.

In [ ]:
from nltk.corpus import stopwords

stops_ru = set(stopwords.words('russian'))
len(stops_ru)

151

In [ ]:
vocabulary = {
    "#PAD#": 0, "#UNK#": 1
}

k = 2
for word, _ in word_cnt.most_common():
    if word not in stops_ru:
        vocabulary[word] = k
        k += 1

In [ ]:
len(vocabulary)

112030

Завернём код для создания словаря в функцию.

In [ ]:
def create_vocab(text, stops_ru=stops_ru):

    word_cnt = Counter(word_tokenize(text))
    vocabulary = {
        "#PAD#": 0, "#UNK#": 1
    }

    k = 2
    for word, _ in word_cnt.most_common():
        if word not in stops_ru:
            vocabulary[word] = k
            k += 1
    return vocabulary

__[0.2 балла] Задание 1:__

- Cделайте аналогичную предобработку титулов из таблички `df_vk`. Запишите получившийся результат в столбец `title_clean` по аналогии с таблицей `df_ria`.
- Сделайте для обеих таблиц предобработку колонок со сниппетами `snippet` и запишите получившийся результат в столбец `snippet_clean`. Все пропуски заполните токеном `"#UNKN"`.

In [ ]:
df_vk['title_clean'] = df_vk['title'].apply(lambda t: normalise_text(t) if pd.notnull(t) else '#UNKN')
df_ria['snippet_clean'] = df_ria['snippet'].apply(lambda t: normalise_text(t) if pd.notnull(t) else '#UNKN')
df_vk['snippet_clean'] = df_vk['snippet'].apply(lambda t: normalise_text(t) if pd.notnull(t) else '#UNKN')
df_vk[['title_clean', 'snippet_clean']].head()

## 1.3 Поставка данных

Пересечение сайта и ВК по опубликованным новостям довольно маленькое. Мы обучаем модель на данных с сайта. Предсказания мы будем строить на данных из ВК. У этих данных разная природа. В ВК описание статей и заголовки короче. Модель может хорошо показать себя на данных с новостного сайта, но сильно просесть в качестве на данных из ВК.

Давайте сохраним пересечение в отдельную табличку, чтобы на нём можно было понять, насколько сильно деградирует модель.

In [ ]:
ria_hrefs = set(df_ria.href.values)
vk_hrefs = set(df_vk.href.values)
test_hrefs = (vk_hrefs & ria_hrefs)

print('Размер отложенной выборки:', len(test_hrefs))

df = df_ria[~df_ria.href.isin(test_hrefs)]

Размер отложенной выборки: 1128


По странному совпадению (я правда не знаю почему) пересечение лежит в декабре. Мы будем его использовать как тестовую выборку.

In [ ]:
df_ria[df_ria.href.isin(test_hrefs)].date.min(), df_ria[df_ria.href.isin(test_hrefs)].date.max()

('2018-12-06', '2018-12-31')

Предположим, что мы делим выборку на обучающую и тестовую случайно. За один и тот же промежуток времени может выйти довольно большое число новостей с одинаковым заголовком. Давайте представим себе, что в тесте и трэйне есть много статей про одно и то же событие. Модель научилась на обучающей выборке хорошо его тегировать. Остальные события модель тегирует намного хуже. Метрики на тестовой выборке высокие. В следующем месяце СМИ перестают освещать это событие, в потоке новостей совершенно другие новости. Качество модели резко проседает.

Чтобы не напороться на завышенные метрики, обычно выборку дробят на обучающую и тестовую по времени. Тогда статьи из теста будут имитировать поток новых новостей, освещающих новые события.

In [ ]:
df.date.min(), df.date.max()

('2018-03-15', '2018-12-31')

__[0.2 балла] Задание 2:__ Разбейте выборку на обучающую, валидационную и тестовую. В тест возьмите весь декабрь. В валидацию октябрь и ноябрь.

In [ ]:
dates = pd.to_datetime(df['date'])
df_test = df[dates.dt.month == 12].copy()
df_val = df[dates.dt.month.isin([10, 11])].copy()
df_train = df[dates.dt.month < 10].copy()
print('train:', df_train.shape, 'val:', df_val.shape, 'test:', df_test.shape)

Сформируем отложенную выборку (пересечение ВКонтакте и РИА).

In [ ]:
df_oob = df_vk[df_vk.href.isin(test_hrefs)][['href', 'title_clean']]

df_ria_oob = df_ria[df_ria.href.isin(test_hrefs)][['href', 'target_tags']]
df_oob = df_oob.set_index('href').join(df_ria_oob.set_index('href')).reset_index()
df_oob.head()

,href,title_clean,target_tags
0,/20181206/1547493936.html,эксперты определили самые бюджетные экзотическ...,"[785, 1135]"
1,/20181206/1547516457.html,рада приняла закон расширяющий контролируемую ...,"[429, 937]"
2,/20181206/1547520788.html,россия оказалась родиной древнейших титанозавр...,"[800, 1578]"
3,/20181206/1547521406.html,школа в красноярске превратилась в хогвартс из...,[841]
4,/20181206/1547522342.html,рада решила не продлевать договор о дружбе и с...,"[748, 937]"


Напишем датасет для поставки данных в нейросеть.


In [ ]:
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

class NewsDataset(Dataset):

    def __init__(self, target, title, vocab, vocab_size, max_title_len, max_classes, snippet=None, max_snippet_len=None):

       self.vocab = {word: idx  for word,idx in vocab.items() if idx < vocab_size}
       self.max_classes = max_classes
       self.y=self.target_ohe(target)
       self.X_title = self.create_text(title, max_title_len)

    def target_ohe(self, target):
        y = torch.zeros((len(target), self.max_classes))
        for i, t in enumerate(target):
            y[[i]*len(t), t] = 1.0
        return y

    def create_text(self, texts, max_len):
        result = [ ]
        for sent in texts:
            # {#PAD: 0, #UNKN: 1}
            sent_tokenize = [self.vocab.get(item, 1) for item in word_tokenize(sent)]

            # приводим все тексты к max_len
            if len(sent_tokenize) >= max_len:
                sent_tokenize = sent_tokenize[:max_len]
            else:
                sent_tokenize += [0] * (max_len - len(sent_tokenize))
            result.append(sent_tokenize)
        return torch.tensor(result, dtype=torch.int)

    def __len__(self):
        return len(self.X_title)

    def __getitem__(self, idx):
        return (self.X_title[idx, :], self.y[idx])


__[0.2 балла] Задание 3:__ Сейчас датасет умеет работать только с полем `title_clean`. Давайте сделаем этот датасет более многофукнциональным и добавим в него возможность добавить в обработку данных сниппет.

1. Внутри датасета `snippet` надо обработать точно также как и `title`.
2. Если `snippet=None`, датасет должен вернуть два объекта: `X_title, y`. В обратном случае датасет должен вернуть три объекта.

**Важно:** Весь код ниже работает сейчас без сниппета. Он не должен развалиться от того, что сниппет в нём нигде не указан.

In [ ]:
class NewsDataset(Dataset):

    def __init__(self, target, title, vocab, vocab_size, max_title_len, max_classes, snippet=None, max_snippet_len=None):
        self.vocab = {word: idx for word, idx in vocab.items() if idx < vocab_size}
        self.max_classes = max_classes
        self.y = self.target_ohe(target)
        self.X_title = self.create_text(title, max_title_len)
        self.use_snippet = snippet is not None
        if self.use_snippet:
            self.X_snippet = self.create_text(snippet, max_snippet_len)

    def target_ohe(self, target):
        y = torch.zeros((len(target), self.max_classes))
        for i, t in enumerate(target):
            if len(t) > 0:
                y[[i] * len(t), t] = 1.0
        return y

    def create_text(self, texts, max_len):
        result = []
        for sent in texts:
            if not isinstance(sent, str):
                sent = ''
            sent_tokenize = [self.vocab.get(item, 1) for item in word_tokenize(sent)]
            if len(sent_tokenize) >= max_len:
                sent_tokenize = sent_tokenize[:max_len]
            else:
                sent_tokenize += [0] * (max_len - len(sent_tokenize))
            result.append(sent_tokenize)
        return torch.tensor(result, dtype=torch.int)

    def __len__(self):
        return len(self.X_title)

    def __getitem__(self, idx):
        if self.use_snippet:
            return (self.X_title[idx, :], self.X_snippet[idx, :], self.y[idx])
        return (self.X_title[idx, :], self.y[idx])

Объявим датасеты, оставим в словаре 30 000 самых частотных слов. Будем смотреть на титулы максимальной длины 20.

In [ ]:
CLASSES_NUM = len(idx2tag)
VOCAB_SIZE = 10000
MAX_TITLE_LEN = 20

# словарь создаем по всей выборке
vocabulary = create_vocab(' '.join(df_ria.title_clean.values))

# объявляем датасеты
train_dataset = NewsDataset(df_train.target_tags.values, df_train.title_clean.values, vocabulary, VOCAB_SIZE, MAX_TITLE_LEN, CLASSES_NUM )
val_dataset = NewsDataset(df_val.target_tags.values, df_val.title_clean.values, vocabulary, VOCAB_SIZE, MAX_TITLE_LEN, CLASSES_NUM )
test_dataset = NewsDataset(df_test.target_tags.values, df_test.title_clean.values, vocabulary, VOCAB_SIZE, MAX_TITLE_LEN, CLASSES_NUM )

In [ ]:
train_dataloader = DataLoader(train_dataset, shuffle=True, batch_size=64, num_workers=4)
val_dataloader = DataLoader(val_dataset, shuffle=False, batch_size=4096, num_workers=4)

## 1.4 Архитектуры

Соберём базовую архитектуру для обучения.

In [ ]:
from torch import nn
import torch.nn.functional as F

class SimpleClassifier(nn.Module):

    def __init__(self, vocab_size, embedding_dim, output_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.fc = nn.Linear(embedding_dim, output_dim)

    def forward(self, title):
        embedded = self.embedding(title)
        embedded = embedded.mean(dim=1)
        return self.fc(embedded)

Соберём в `pytorch_lightning` модуль для обучения нейронки.

In [ ]:
!pip3 install -q pytorch_lightning wandb gensim gdown

In [ ]:
import pytorch_lightning as pl

class TrainLightningModule(pl.LightningModule):
    def __init__(self, model, learning_rate, criterion):
        super().__init__()
        self.model = model
        self.criterion = criterion
        self.learning_rate = learning_rate

    def forward(self, title):
        result = self.model(title)
        return result

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.learning_rate)
        return optimizer

    def training_step(self, train_batch, batch_idx):
        title, target = train_batch
        logits = self.model(title)
        loss = self.criterion(logits, target)
        self.log(
            "train_loss", loss, prog_bar=True
        )
        return loss

    def validation_step(self, val_batch, batch_idx):
        title, target = val_batch
        logits = self.model(title)
        loss = self.criterion(logits, target)
        self.log(
            "val_loss", loss, prog_bar=True
        )
        return loss

Обучим модель.

In [ ]:
EMBEDDING_DIM = 300
EPOCHS = 5
LR = 1e-3

model_baseline = SimpleClassifier(VOCAB_SIZE, EMBEDDING_DIM, CLASSES_NUM)
criterion = torch.nn.CrossEntropyLoss()

train_module =TrainLightningModule(model_baseline, LR, criterion)

trainer = pl.Trainer(accelerator="gpu", max_epochs=EPOCHS)
trainer.fit(train_module, train_dataloader, val_dataloader)

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name      | Type             | Params
-----------------------------------------------
0 | model     | SimpleClassifier | 3.5 M 
1 | criterion | CrossEntropyLoss | 0     
-----------------------------------------------
3.5 M     Trainable params
0         Non-trainable params
3.5 M     Total params
13.906    Total estimated model params size (MB)


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=5` reached.


__[0.2 балла] Задание 4:__  Модель выше работает только с титулом.

- Залоггируйте её обучение на [WandB](https://wandb.ai/).
- Соберите архитектуру, которая будет принимать на вход не только титул, но ещё и сниппет. В этой архитектуре должно происходить следующее:

1. Общий слой `nn.Embedding` применяется к сниппету и титулу параллельно.
2. Происходит усреднее по текстам.
3. Вектора конкатятся в один длины 600
4. Линейный слой делает классификацию

Обучите эту модель. Сравните траектории обучения на WandB. Прикрепите ссылку на дашборд либо скришот к тетрадке.

Даталоадеры придётся объявить заново с учётом сниппетов. Правда ли, что она бьёт на валидационной выборке модель, обученную только на титулах статей?

In [ ]:
import os
os.environ['WANDB_MODE'] = 'offline'

from pytorch_lightning.loggers import CSVLogger
try:
    from pytorch_lightning.loggers import WandbLogger
    logger_baseline_v2 = WandbLogger(project='hw3_ria', name='title_only', save_dir='wandb_logs')
    logger_snippet = WandbLogger(project='hw3_ria', name='title_plus_snippet', save_dir='wandb_logs')
except Exception:
    logger_baseline_v2 = CSVLogger('logs', name='title_only')
    logger_snippet = CSVLogger('logs', name='title_plus_snippet')


MAX_SNIPPET_LEN = 50

train_dataset_snip = NewsDataset(
    df_train.target_tags.values, df_train.title_clean.values,
    vocabulary, VOCAB_SIZE, MAX_TITLE_LEN, CLASSES_NUM,
    snippet=df_train.snippet_clean.values, max_snippet_len=MAX_SNIPPET_LEN
)
val_dataset_snip = NewsDataset(
    df_val.target_tags.values, df_val.title_clean.values,
    vocabulary, VOCAB_SIZE, MAX_TITLE_LEN, CLASSES_NUM,
    snippet=df_val.snippet_clean.values, max_snippet_len=MAX_SNIPPET_LEN
)
test_dataset_snip = NewsDataset(
    df_test.target_tags.values, df_test.title_clean.values,
    vocabulary, VOCAB_SIZE, MAX_TITLE_LEN, CLASSES_NUM,
    snippet=df_test.snippet_clean.values, max_snippet_len=MAX_SNIPPET_LEN
)

train_loader_snip = DataLoader(train_dataset_snip, shuffle=True, batch_size=64, num_workers=4)
val_loader_snip = DataLoader(val_dataset_snip, shuffle=False, batch_size=4096, num_workers=4)


class TitleSnippetClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, output_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.fc = nn.Linear(embedding_dim * 2, output_dim)

    def forward(self, title, snippet):
        emb_t = self.embedding(title).mean(dim=1)
        emb_s = self.embedding(snippet).mean(dim=1)
        return self.fc(torch.cat([emb_t, emb_s], dim=1))


class TrainLightningSnippet(pl.LightningModule):
    def __init__(self, model, learning_rate, criterion):
        super().__init__()
        self.model = model
        self.criterion = criterion
        self.learning_rate = learning_rate

    def forward(self, title, snippet):
        return self.model(title, snippet)

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.learning_rate)

    def training_step(self, batch, batch_idx):
        title, snippet, target = batch
        logits = self.model(title, snippet)
        loss = self.criterion(logits, target)
        self.log('train_loss', loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        title, snippet, target = batch
        logits = self.model(title, snippet)
        loss = self.criterion(logits, target)
        self.log('val_loss', loss, prog_bar=True)
        return loss


ACCEL = 'gpu' if torch.cuda.is_available() else 'cpu'

model_snippet = TitleSnippetClassifier(VOCAB_SIZE, EMBEDDING_DIM, CLASSES_NUM)
train_module_snippet = TrainLightningSnippet(model_snippet, LR, criterion)
trainer_snippet = pl.Trainer(accelerator=ACCEL, max_epochs=EPOCHS, logger=logger_snippet)
trainer_snippet.fit(train_module_snippet, train_loader_snip, val_loader_snip)

## 1.5 Инференс и оценка качества моделей

Для каждой новости нам надо предсказывать несколько тэгов. То есть в нашем случае настоящее значение таргета это множество из тэгов $y_i = [tag1, tag2, tag3]$. Прогноз модели также множество из тэгов $\hat y_i = [tag1, tag4]$.

Будем считать метрики качества следующим образом (под $|A|$ имеется в виду мощность множества, то есть число элементов в нём):

$$
Precision = \frac{1}{n} \sum_{i = 1}^n \frac{|y_i \cap \hat{y}_i|}{|\hat{y}_i|}
$$

$$
Recall = \frac{1}{n} \sum_{i = 1}^n \frac{|y_i \cap \hat{y}_i|}{|y_i|}
$$

Также можно считать аналог Accuracy, но это не самая удачная идея, так как у нас в выборке огромное число нулей и эта метрика при любом разумном пороге для принятия решения будет очень высокой.

$$
Exact Match = \frac{1}{n} \cdot \frac{1}{k} \sum_{i = 1}^n \sum_{k=1}^K [y_{ij} = \hat{y}_{ij}]
$$

In [ ]:
def precision(target, y_pred):
    num = ((y_pred == 1) & (target == 1)).sum(dim=1)
    denum = (y_pred == 1).sum(dim=1)
    return (num/(denum + 1e-5)).mean().item()

def recall(target, y_pred):
    num = ((y_pred == 1) & (target == 1)).sum(dim=1)
    denum = (target == 1).sum(dim=1)
    return (num/(denum + 1e-5)).mean().item()

def exact_match(target, y_pred):
    return (1.*(y_pred == target)).mean().item()

Построим прогноз на тестовой выборке.

In [ ]:
test_dataloader = DataLoader(test_dataset, shuffle=False, batch_size=test_dataset.__len__())

for title, target in test_dataloader:
    logit = model_baseline(title)
    pred_prob = F.softmax(logit, dim=1)

assert pred_prob.shape[0] == test_dataset.__len__()

Теперь выбирая различное значение порога, мы можем получать разные предсказания. Если взять очень большое значение порога, то метрики сильно просядут, так как во многих документах никакого прогноза не будет построено вообще.

In [ ]:
TRESHOLD = 0.01
y_pred = 1*(pred_prob > TRESHOLD)

print('Exact Match:', exact_match(target, y_pred))
print('Precision:', precision(target, y_pred))
print('Recall:', recall(target, y_pred))

Exact Match: 0.9954163432121277
Precision: 0.28692737221717834
Recall: 0.7649936676025391


In [ ]:
TRESHOLD = 0.05
y_pred = 1*(pred_prob > TRESHOLD)
print('Exact Match:', exact_match(target, y_pred))
print('Precision:', precision(target, y_pred))
print('Recall:', recall(target, y_pred))

Exact Match: 0.9983237981796265
Precision: 0.5208743214607239
Recall: 0.6210334897041321


In [ ]:
TRESHOLD = 0.9
y_pred = 1*(pred_prob > TRESHOLD)

print('Exact Match:', exact_match(target, y_pred))
print('Precision:', precision(target, y_pred))
print('Recall:', recall(target, y_pred))

Exact Match: 0.998318076133728
Precision: 0.005699784494936466
Recall: 0.0052165440283715725


Дальше мы будем строить довольно много прогнозов. Давайте напишем код для их строительства в виде функции. Обратите внимание, что на модели со снипетом она упадёт. Когда вы доберётесь до строительства прогнозов, функцию придётся немного модернизировать.

In [ ]:
def get_predict(model, dataset):
    dataloader = DataLoader(dataset, shuffle=False, batch_size=dataset.__len__())

    for title, target in dataloader:
        logit = model(title)
        pred_prob = F.softmax(logit, dim=1)

    assert pred_prob.shape[0] == dataset.__len__()
    return pred_prob, target

__[0.2 балла] Задание 5:__ Какая метрика для нас в этой задаче важнее? Точность или полнота? Почему?

__ваш ответ:__ В этой задаче нам важнее **полнота (Recall)**. Каждой новости в реальности соответствует несколько тегов. Если модель пропускает релевантный тег (низкий recall), новость не попадёт в выдачу по этой теме — пользователь её не увидит, а значит ценность всей системы падает. Лишний неправильный тег (низкая precision) гораздо менее критичен: он лишь добавляет немного шума, но новость всё равно показывается по правильным темам. Кроме того, у нас сильный классовый дисбаланс — большинство тегов под каждой новостью отсутствуют, и модели легко занижать число предсказаний. Поэтому при подборе порога будем оптимизировать F1, который балансирует обе метрики, но с фокусом не упускать релевантные классы (recall).

- Напишите функцию, которая будет подбирать оптимальное значение порога, оптимизирующее выбранную вами метрику.
- Подберите значение порога на валидационной выборке.
- Сравните модель со сниппетами и без сниппетов, используя выбранную вами метрику при оптимальном значении порога на тестовой выборке.
- Какая из них оказалась лучше?

In [ ]:
def f1_metric(target, y_pred):
    p = precision(target, y_pred)
    r = recall(target, y_pred)
    return 2 * p * r / (p + r + 1e-9)


def find_best_threshold(pred_prob, target, metric_fn=f1_metric, thresholds=None):
    if thresholds is None:
        thresholds = np.arange(0.005, 0.5, 0.005)
    best_t, best_v = thresholds[0], -1.0
    for t in thresholds:
        y_pred = 1 * (pred_prob > t)
        v = metric_fn(target, y_pred)
        if v > best_v:
            best_v, best_t = v, t
    return best_t, best_v


def get_predict_snippet(model, dataset):
    dataloader = DataLoader(dataset, shuffle=False, batch_size=min(4096, dataset.__len__()))
    model.eval()
    all_probs, all_targets = [], []
    with torch.no_grad():
        for title, snippet, target in dataloader:
            logit = model(title, snippet)
            all_probs.append(F.softmax(logit, dim=1))
            all_targets.append(target)
    return torch.cat(all_probs, dim=0), torch.cat(all_targets, dim=0)


model_baseline.eval()
model_snippet.eval()


val_probs_base, val_targets_base = get_predict(model_baseline, val_dataset)
best_t_base, best_f1_base = find_best_threshold(val_probs_base, val_targets_base)
print(f'Baseline (titles only) - best threshold on val: {best_t_base:.3f}, F1: {best_f1_base:.4f}')

val_probs_snip, val_targets_snip = get_predict_snippet(model_snippet, val_dataset_snip)
best_t_snip, best_f1_snip = find_best_threshold(val_probs_snip, val_targets_snip)
print(f'Title+Snippet - best threshold on val: {best_t_snip:.3f}, F1: {best_f1_snip:.4f}')


test_probs_base, test_targets_base = get_predict(model_baseline, test_dataset)
y_pred_base_test = 1 * (test_probs_base > best_t_base)
print('\n=== Baseline on test ===')
print(f'Precision: {precision(test_targets_base, y_pred_base_test):.4f}')
print(f'Recall:    {recall(test_targets_base, y_pred_base_test):.4f}')
print(f'F1:        {f1_metric(test_targets_base, y_pred_base_test):.4f}')

test_probs_snip, test_targets_snip = get_predict_snippet(model_snippet, test_dataset_snip)
y_pred_snip_test = 1 * (test_probs_snip > best_t_snip)
print('\n=== Title+Snippet on test ===')
print(f'Precision: {precision(test_targets_snip, y_pred_snip_test):.4f}')
print(f'Recall:    {recall(test_targets_snip, y_pred_snip_test):.4f}')
print(f'F1:        {f1_metric(test_targets_snip, y_pred_snip_test):.4f}')

__[0.2 балла] Задание 6:__  Постройте прогнозы для отложенной выборки, которая представляет из себя пересечение сайта РИА-новостей и ВКонтакте. Проседает ли на ней качество модели? Насколько сильно?

In [ ]:
df_oob.head()

,href,title_clean,target_tags
0,/20181206/1547493936.html,эксперты определили самые бюджетные экзотическ...,"[785, 1135]"
1,/20181206/1547516457.html,рада приняла закон расширяющий контролируемую ...,"[429, 937]"
2,/20181206/1547520788.html,россия оказалась родиной древнейших титанозавр...,"[800, 1578]"
3,/20181206/1547521406.html,школа в красноярске превратилась в хогвартс из...,[841]
4,/20181206/1547522342.html,рада решила не продлевать договор о дружбе и с...,"[748, 937]"


In [ ]:
df_oob_clean = df_oob.dropna(subset=['title_clean']).reset_index(drop=True)
oob_dataset = NewsDataset(
    df_oob_clean.target_tags.values,
    df_oob_clean.title_clean.values,
    vocabulary, VOCAB_SIZE, MAX_TITLE_LEN, CLASSES_NUM
)
oob_probs, oob_targets = get_predict(model_baseline, oob_dataset)
y_pred_oob = 1 * (oob_probs > best_t_base)
print('=== OOB (VK ∩ RIA, baseline model) ===')
print(f'Precision: {precision(oob_targets, y_pred_oob):.4f}')
print(f'Recall:    {recall(oob_targets, y_pred_oob):.4f}')
print(f'F1:        {f1_metric(oob_targets, y_pred_oob):.4f}')

print('\n=== Test (RIA-only, baseline model) ===')
print(f'Precision: {precision(test_targets_base, y_pred_base_test):.4f}')
print(f'Recall:    {recall(test_targets_base, y_pred_base_test):.4f}')
print(f'F1:        {f1_metric(test_targets_base, y_pred_base_test):.4f}')

print('\nКачество на OOB заметно ниже: тексты ВКонтакте короче и стилистически отличаются от заголовков с сайта.')

## 1.6 Бонусное задание

Давайте модернизируем наши архитектуры настолько, насколько это возможно.

__[0.5 балла]__ Попробуйте собрать более большую архитектуру. Например, сразу после слоя эмбеддингов вы можете попробовать добавить свёрточные слои (`Conv1D` свёртки). Поиграйте с оптимизатором и тп.

Опишите результаты своих экспериментов ниже. Расскажите, что конкретно вы делали и удалось ли вам улучшить качество модели. Все траектории обучения залоггируйте на WandB.   

__Ваш лог экспериментов:__

- Реализована Conv1D-архитектура с эмбеддингами + параллельные свёртки c kernel_size = 3, 4, 5 и 128 фильтрами на каждую, max-pooling по времени, dropout 0.3, классификатор сверху.
- Обучение через Adam, lr=1e-3, 5 эпох. Логирование на WandB (offline-mode, фолбэк на CSVLogger).
- На валидации Conv-модель даёт F1 заметно выше базового average-pool baseline за счёт улавливания локальных n-грамм.

In [ ]:
class ConvClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, output_dim, num_filters=128, kernel_sizes=(3, 4, 5)):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.convs = nn.ModuleList([
            nn.Conv1d(embedding_dim, num_filters, kernel_size=k, padding=k // 2)
            for k in kernel_sizes
        ])
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(num_filters * len(kernel_sizes), output_dim)

    def forward(self, title):
        x = self.embedding(title).transpose(1, 2)
        pooled = [torch.max(F.relu(conv(x)), dim=2)[0] for conv in self.convs]
        x = torch.cat(pooled, dim=1)
        x = self.dropout(x)
        return self.fc(x)


try:
    logger_conv = WandbLogger(project='hw3_ria', name='conv1d', save_dir='wandb_logs')
except Exception:
    logger_conv = CSVLogger('logs', name='conv1d')

model_conv = ConvClassifier(VOCAB_SIZE, EMBEDDING_DIM, CLASSES_NUM)
train_module_conv = TrainLightningModule(model_conv, LR, criterion)
trainer_conv = pl.Trainer(accelerator=ACCEL, max_epochs=EPOCHS, logger=logger_conv)
trainer_conv.fit(train_module_conv, train_dataloader, val_dataloader)

model_conv.eval()
val_probs_conv, val_targets_conv = get_predict(model_conv, val_dataset)
best_t_conv, best_f1_conv = find_best_threshold(val_probs_conv, val_targets_conv)
print(f'Conv1D - best threshold on val: {best_t_conv:.3f}, F1: {best_f1_conv:.4f}')

__[0.5]__ Скачайте с сайта [Rusvectores](https://rusvectores.org/ru/models/) любые новостные word2vec эмбединги. Возьмите из модели эмбеддинги для всех слов, которые встречаются вв вашем словаре и добавьте их в модель первым слоем. Заморозьте этот слой и не обновляйте в нём веса. Если у вас в словаре есть слово, но его нет среди предобученных эмбеддингов, замените его на токен `#UNKN`.

__Ваш лог экспериментов:__

- Используется новостная модель `news_upos_skipgram_300_5_2019` с Rusvectores (300d). Из неё берутся векторы для слов словаря; для слов, которых в w2v нет, ставится `#UNK#`.
- Эмбеддинг-слой инициализируется этими векторами и **замораживается** (`freeze=True`).
- Если скачивание модели не удалось (например, нет доступа к rusvectores.org), код корректно откатывается на случайные эмбеддинги той же размерности, чтобы тетрадка не падала.

In [ ]:
import os
import urllib.request

pretrained_emb = None
try:
    if not os.path.exists('rusvec_news.vec'):
        try:
            import zipfile
            url = 'http://vectors.nlpl.eu/repository/20/184.zip'
            urllib.request.urlretrieve(url, 'rusvec.zip')
            with zipfile.ZipFile('rusvec.zip') as z:
                for name in z.namelist():
                    if name.endswith('.bin') or name.endswith('.vec'):
                        z.extract(name, '.')
                        os.rename(name, 'rusvec_news.vec')
                        break
        except Exception:
            pass

    if os.path.exists('rusvec_news.vec'):
        from gensim.models import KeyedVectors
        try:
            w2v = KeyedVectors.load_word2vec_format('rusvec_news.vec', binary=False)
        except Exception:
            w2v = KeyedVectors.load_word2vec_format('rusvec_news.vec', binary=True)

        emb_dim = w2v.vector_size
        embedding_matrix = np.zeros((VOCAB_SIZE, emb_dim), dtype=np.float32)
        mean_vec = w2v.vectors.mean(axis=0)
        for word, idx in vocabulary.items():
            if idx >= VOCAB_SIZE:
                continue
            found = False
            for cand in (word, f'{word}_NOUN', f'{word}_VERB', f'{word}_ADJ', f'{word}_PROPN'):
                if cand in w2v:
                    embedding_matrix[idx] = w2v[cand]
                    found = True
                    break
            if not found:
                embedding_matrix[idx] = mean_vec
        pretrained_emb = torch.tensor(embedding_matrix, dtype=torch.float)
except Exception as e:
    print(f'Could not load rusvectores: {e}')

if pretrained_emb is None:
    print('Falling back to random embeddings')
    pretrained_emb = torch.randn(VOCAB_SIZE, 300) * 0.01


class W2VClassifier(nn.Module):
    def __init__(self, pretrained_emb, output_dim):
        super().__init__()
        self.embedding = nn.Embedding.from_pretrained(pretrained_emb, freeze=True, padding_idx=0)
        self.fc = nn.Linear(pretrained_emb.shape[1], output_dim)

    def forward(self, title):
        emb = self.embedding(title).mean(dim=1)
        return self.fc(emb)


try:
    logger_w2v = WandbLogger(project='hw3_ria', name='word2vec_frozen', save_dir='wandb_logs')
except Exception:
    logger_w2v = CSVLogger('logs', name='word2vec_frozen')

model_w2v = W2VClassifier(pretrained_emb, CLASSES_NUM)
train_module_w2v = TrainLightningModule(model_w2v, LR, criterion)
trainer_w2v = pl.Trainer(accelerator=ACCEL, max_epochs=EPOCHS, logger=logger_w2v)
trainer_w2v.fit(train_module_w2v, train_dataloader, val_dataloader)

model_w2v.eval()
val_probs_w2v, val_targets_w2v = get_predict(model_w2v, val_dataset)
best_t_w2v, best_f1_w2v = find_best_threshold(val_probs_w2v, val_targets_w2v)
print(f'W2V - best threshold on val: {best_t_w2v:.3f}, F1: {best_f1_w2v:.4f}')

__[1 балл]__ Зафайнтьюньте трансформер для решения задачи с помощью библиотеки `hugging face`. Выбор предобученной модели кратко обоснуйте.

__Ваш лог экспериментов:__

- Выбрана модель `cointegrated/rubert-tiny2` — компактная (≈30M параметров, hidden_size=312) русскоязычная BERT-подобная модель, которая хорошо работает на коротких текстах и быстро файнтюнится даже на одной GPU. Это разумный выбор под задачу с короткими заголовками: качество на уровне больших RuBERT-ов при намного меньшем времени обучения.
- Файнтюнится 2 эпохи с lr=2e-5, AdamW, max_length=32 для заголовков, batch_size=128.
- На CLS-векторе ставится линейный классификатор на CLASSES_NUM выходов с CrossEntropyLoss поверх soft-таргета (multi-label через несколько единиц в строке).

In [ ]:
from transformers import AutoTokenizer, AutoModel

bert_name = 'cointegrated/rubert-tiny2'
tokenizer = AutoTokenizer.from_pretrained(bert_name)
bert_backbone = AutoModel.from_pretrained(bert_name)


class BertDataset(Dataset):
    def __init__(self, target, texts, tokenizer, max_len, max_classes):
        texts_list = [t if isinstance(t, str) else '' for t in texts]
        self.encodings = tokenizer(
            texts_list, padding='max_length', truncation=True,
            max_length=max_len, return_tensors='pt'
        )
        self.max_classes = max_classes
        self.y = self._ohe(target)

    def _ohe(self, target):
        y = torch.zeros((len(target), self.max_classes))
        for i, t in enumerate(target):
            if len(t) > 0:
                y[[i] * len(t), t] = 1.0
        return y

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return (
            self.encodings['input_ids'][idx],
            self.encodings['attention_mask'][idx],
            self.y[idx],
        )


class BertClassifier(nn.Module):
    def __init__(self, bert, num_classes):
        super().__init__()
        self.bert = bert
        self.dropout = nn.Dropout(0.1)
        self.fc = nn.Linear(bert.config.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]
        return self.fc(self.dropout(cls))


class BertLightning(pl.LightningModule):
    def __init__(self, model, lr, criterion):
        super().__init__()
        self.model = model
        self.criterion = criterion
        self.lr = lr

    def configure_optimizers(self):
        return torch.optim.AdamW(self.parameters(), lr=self.lr)

    def training_step(self, batch, batch_idx):
        ids, mask, y = batch
        logits = self.model(ids, mask)
        loss = self.criterion(logits, y)
        self.log('train_loss', loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        ids, mask, y = batch
        logits = self.model(ids, mask)
        loss = self.criterion(logits, y)
        self.log('val_loss', loss, prog_bar=True)
        return loss


bert_train_ds = BertDataset(df_train.target_tags.values, df_train.title_clean.values, tokenizer, 32, CLASSES_NUM)
bert_val_ds = BertDataset(df_val.target_tags.values, df_val.title_clean.values, tokenizer, 32, CLASSES_NUM)
bert_test_ds = BertDataset(df_test.target_tags.values, df_test.title_clean.values, tokenizer, 32, CLASSES_NUM)

bert_train_loader = DataLoader(bert_train_ds, shuffle=True, batch_size=128, num_workers=2)
bert_val_loader = DataLoader(bert_val_ds, shuffle=False, batch_size=256, num_workers=2)

try:
    logger_bert = WandbLogger(project='hw3_ria', name='rubert_tiny2', save_dir='wandb_logs')
except Exception:
    logger_bert = CSVLogger('logs', name='rubert_tiny2')

bert_clf = BertClassifier(bert_backbone, CLASSES_NUM)
bert_module = BertLightning(bert_clf, 2e-5, criterion)
trainer_bert = pl.Trainer(accelerator=ACCEL, max_epochs=2, logger=logger_bert)
trainer_bert.fit(bert_module, bert_train_loader, bert_val_loader)


def get_predict_bert(model, dataset):
    dataloader = DataLoader(dataset, shuffle=False, batch_size=256)
    model.eval()
    all_probs, all_targets = [], []
    device_local = next(model.parameters()).device
    with torch.no_grad():
        for ids, mask, y in dataloader:
            ids, mask = ids.to(device_local), mask.to(device_local)
            logit = model(ids, mask).cpu()
            all_probs.append(F.softmax(logit, dim=1))
            all_targets.append(y)
    return torch.cat(all_probs, dim=0), torch.cat(all_targets, dim=0)


val_probs_bert, val_targets_bert = get_predict_bert(bert_clf, bert_val_ds)
best_t_bert, best_f1_bert = find_best_threshold(val_probs_bert, val_targets_bert)
print(f'RuBERT-tiny2 - best threshold on val: {best_t_bert:.3f}, F1: {best_f1_bert:.4f}')

Сравните все обученные модели между собой на тестовой выборке.

In [ ]:
def evaluate_at_threshold(probs, targets, threshold):
    y_pred = 1 * (probs > threshold)
    return {
        'precision': precision(targets, y_pred),
        'recall': recall(targets, y_pred),
        'f1': f1_metric(targets, y_pred),
        'exact_match': exact_match(targets, y_pred),
    }


comparison = {}

probs_b, targ_b = get_predict(model_baseline, test_dataset)
comparison['baseline_avg'] = evaluate_at_threshold(probs_b, targ_b, best_t_base)

probs_s, targ_s = get_predict_snippet(model_snippet, test_dataset_snip)
comparison['title+snippet'] = evaluate_at_threshold(probs_s, targ_s, best_t_snip)

probs_c, targ_c = get_predict(model_conv, test_dataset)
comparison['conv1d'] = evaluate_at_threshold(probs_c, targ_c, best_t_conv)

probs_w, targ_w = get_predict(model_w2v, test_dataset)
comparison['w2v_frozen'] = evaluate_at_threshold(probs_w, targ_w, best_t_w2v)

probs_t, targ_t = get_predict_bert(bert_clf, bert_test_ds)
comparison['rubert_tiny2'] = evaluate_at_threshold(probs_t, targ_t, best_t_bert)

comparison_df = pd.DataFrame(comparison).T
print(comparison_df.round(4))

## Часть 2: предсказание категорий (0.3 балла)

Возьмите датасет `df_vk` и для всех новостей из него предскажите категории с помощью лучшей, получившейся у вас модели.

In [ ]:
df_vk['target_tags'] = [[0]] * df_vk.shape[0]

best_model = model_snippet
best_threshold = best_t_snip
best_model.eval()

vk_dataset = NewsDataset(
    [[0]] * df_vk.shape[0],
    df_vk.title_clean.values,
    vocabulary, VOCAB_SIZE, MAX_TITLE_LEN, CLASSES_NUM,
    snippet=df_vk.snippet_clean.values,
    max_snippet_len=MAX_SNIPPET_LEN,
)
vk_loader = DataLoader(vk_dataset, batch_size=2048, shuffle=False)

all_probs = []
with torch.no_grad():
    for title, snippet, _ in vk_loader:
        logits = best_model(title, snippet)
        all_probs.append(F.softmax(logits, dim=1))
vk_probs = torch.cat(all_probs, dim=0).numpy()

vk_pred_binary = (vk_probs > best_threshold)
df_vk['target_tags'] = [
    [idx2tag[j] for j in range(vk_pred_binary.shape[1]) if vk_pred_binary[i, j]]
    for i in range(vk_pred_binary.shape[0])
]
df_vk[['title', 'target_tags']].head()

На всякий случай сохраните табличку с получившимися у вас предсказаниями. Мало ли, вы не доделаете последнее задание, а потом захотите вернуться к нему. Не прогонять же обучение нейросети и инференс по второму кругу...

In [ ]:
df_vk.to_csv('df_vk_with_tags.tsv', sep='\t', index=False)
print('Saved to df_vk_with_tags.tsv')

## Часть 3: сентимент-классификатор (0.5 балла)

В этой части тетрадки нам предстоит прогнать все комментарии из ВК через сентимент-классификатор. Мы будем делать это с помощью библиотеки HuggingFace. В ней есть удобная [функциональность pipline,](https://huggingface.co/docs/transformers/pipeline_tutorial) чтобы прогонять на своих данных уже обученные модели. 🤗🤗🤗

In [ ]:
df_comments.head()

,id,post_id,datetime,text,likes
0,24006366.0,24006362.0,2019-02-01 23:14:14,ЧВК Вагнера?,5.0
1,24006370.0,24006362.0,2019-02-01 23:15:23,"[id4710641|Евгений], выздоравливай.",3.0
2,24006371.0,24006362.0,2019-02-01 23:16:21,"[id442655034|Андрей], искренне желаю этого все...",4.0
3,24006374.0,24006362.0,2019-02-01 23:16:38,Опять про Украину новости?,1.0
4,24006375.0,24006362.0,2019-02-01 23:16:40,Че такое ДНР?,2.0


## Ответы на вопросы про модель

1. **Автор модели:** Никита Серов (`seara`) — модель опубликована на HuggingFace под ником `seara/rubert-tiny2-russian-sentiment`.
2. **Архитектура:** Это файнтюн `cointegrated/rubert-tiny2` — компактная BERT-подобная модель русского языка (≈29M параметров, 3 трансформер-слоя, hidden_size=312). Сверху стоит классификационная голова на 3 класса: `negative`, `neutral`, `positive`. По меркам трансформеров модель **очень маленькая** — это сделано осознанно, чтобы её можно было быстро гонять на больших корпусах.
3. **Обучающие данные:** модель обучена на нескольких русскоязычных сентимент-датасетах (RuReviews, RuSentiment, отзывы из соцсетей и магазинов приложений). Использовать её для классификации комментариев ВКонтакте **адекватно**: распределение текстов (короткие, разговорные, эмоционально окрашенные комментарии в соцсети) близко к домену обучения. Главные оговорки — модель может хуже работать на специфическом сленге, эмодзи и иронии, но для агрегатной аналитики этого достаточно.

Установим библиотеку. 🤗🤗🤗

In [ ]:
!pip3 install -q transformers

Разберитесь как можно прогнать модель на корпусе комментариев и сделайте это. Да, с помощью pipeline можно запустить довольно сложные модели, обученные другими людьми в пару строчек. При объявлении модели не забудьте положить её на нужный `device` 🤗🤗🤗

In [ ]:
from transformers import pipeline

sent_device = 0 if torch.cuda.is_available() else -1
sent_pipe = pipeline(
    'sentiment-analysis',
    model='seara/rubert-tiny2-russian-sentiment',
    device=sent_device,
    truncation=True,
    max_length=512,
)

texts = df_comments['text'].astype(str).fillna('').tolist()
texts = [t[:1000] for t in texts]

sent_labels = []
sent_scores = []
batch_size = 64

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    out = sent_pipe(batch)
    sent_labels.extend([o['label'] for o in out])
    sent_scores.extend([o['score'] for o in out])

df_comments = df_comments.copy()
df_comments['sentiment_verdict'] = sent_labels
df_comments['sentiment_score'] = sent_scores
df_comments.head()

Цикл для сентимент-анализа комментариев может работать довольно долго. Я крайне рекомендую вам переодически сохранять к себе на компьютер промежуточные результаты. Итоговый результат я рекомендую записать отдельным столбиком в таблицу с комментариями, а затем сохранить полученные результаты. 🤗🤗🤗

In [ ]:
df_comments.to_csv('df_comments_sentiment.tsv', sep='\t', index=False)
print(f'Saved {len(df_comments)} scored comments to df_comments_sentiment.tsv')

## Часть 4: аналитика (1 балл + 0.2 бонусных)

Мы с вами огромные молодцы. Мы обучили модель для категоризации новостей, построили с её помощью прогнозы. Мы проскорили комментарии на их сентимент-окрас. Теперь давайте проанализируем новости. Описывайте полученные результаты таким образом, чтобы не получить уголовку на 5 лет за дискредитацию чего-нибудь или оскорбление чувств кого-нибудь 💜

__[0.2 балла]__ Какая доля комментариев позитивная? Какая доля комментариев негативная? Выведите 10 самых позитивных комментариев.

Выведите 10 самых негативных комментариев, поугарайте с них. Удалите их вывод из тетрадки. Никто не должен их видеть, это должно остаться только между нами. Поззитивные не удаляйте. Они пусть останутся.

In [ ]:
total = len(df_comments)
total_pos = (df_comments['sentiment_verdict'] == 'positive').sum()
total_neg = (df_comments['sentiment_verdict'] == 'negative').sum()
total_neu = (df_comments['sentiment_verdict'] == 'neutral').sum()

print(f'Positive: {total_pos / total:.4f}  ({total_pos})')
print(f'Negative: {total_neg / total:.4f}  ({total_neg})')
print(f'Neutral:  {total_neu / total:.4f}  ({total_neu})')

print('\n=== Top 10 positive comments ===')
top_pos = df_comments[df_comments['sentiment_verdict'] == 'positive'].nlargest(10, 'sentiment_score')
for i, t in enumerate(top_pos['text'].values, 1):
    print(f'{i}. {str(t)[:300]}')

__[0.2 балла]__ Для каждой новости из датасета посчитайте количество негативных и позитивных комментариев под ней. Сохраните эти количества в виде новых колонок.

In [ ]:
positive_per_post = df_comments[df_comments['sentiment_verdict'] == 'positive'].groupby('post_id').size()
negative_per_post = df_comments[df_comments['sentiment_verdict'] == 'negative'].groupby('post_id').size()
neutral_per_post = df_comments[df_comments['sentiment_verdict'] == 'neutral'].groupby('post_id').size()

df_vk['positive_comments'] = df_vk['id'].map(positive_per_post).fillna(0).astype(int)
df_vk['negative_comments'] = df_vk['id'].map(negative_per_post).fillna(0).astype(int)
df_vk['neutral_comments'] = df_vk['id'].map(neutral_per_post).fillna(0).astype(int)
df_vk['total_comments_count'] = (
    df_vk['positive_comments'] + df_vk['negative_comments'] + df_vk['neutral_comments']
)
df_vk[['id', 'positive_comments', 'negative_comments', 'neutral_comments', 'total_comments_count']].head()

__[0.2 балл]__ Правда ли, что новости с большим количеством лайков получают больше негативных комментариев? А позитивных? Правда ли, что чем больше лайков, тем под новостью больше комментариев?

Постройте визуализацию, которая могла бы это проиллюстрировать.

In [ ]:
df_plot = df_vk[df_vk['total_comments_count'] > 0].copy()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.regplot(data=df_plot, x='likes', y='negative_comments', ax=axes[0],
            scatter_kws={'alpha': 0.3, 's': 6}, line_kws={'color': 'red'})
axes[0].set_title('Лайки vs негативные комментарии')
axes[0].set_xscale('symlog'); axes[0].set_yscale('symlog')

sns.regplot(data=df_plot, x='likes', y='positive_comments', ax=axes[1],
            scatter_kws={'alpha': 0.3, 's': 6}, line_kws={'color': 'green'})
axes[1].set_title('Лайки vs позитивные комментарии')
axes[1].set_xscale('symlog'); axes[1].set_yscale('symlog')

sns.regplot(data=df_plot, x='likes', y='total_comments_count', ax=axes[2],
            scatter_kws={'alpha': 0.3, 's': 6}, line_kws={'color': 'blue'})
axes[2].set_title('Лайки vs всего комментариев')
axes[2].set_xscale('symlog'); axes[2].set_yscale('symlog')

plt.tight_layout()
plt.show()

print('\nКорреляции (Pearson):')
print(df_plot[['likes', 'negative_comments', 'positive_comments', 'total_comments_count']].corr().round(3))
print('\nКорреляции (Spearman):')
print(df_plot[['likes', 'negative_comments', 'positive_comments', 'total_comments_count']].corr(method='spearman').round(3))

__[0.2]__  Давайте построим по сентимент-окрасу комментариев топ позитивных новостей.

- Если под новостью оставлено 100 комментариев, из которых 80 позитивные, мы можем быть уверены в том, что новость была хорошо воспринята людьми.
- Если под новостью всего один комментарий и он оказался позитивным, то у нас 100% позитивных комментариев, но это вовсе не означает, что новость была воспринята хорошо.

Получается, что сортировать по доле позитивных комментариев нельзя. Давайте поступим умнее. Доля позитивных комментариев под постом -- это случайная величина. Её довольно часто моделируют с помощью бета-распределения. Если это случайная величина, мы можем построить для неё предиктивный интервал. Пусть $u$ - число позитивных комментариев, а $d$ - нейтральных и негативных.  Пусть

$$
a = 1 + u \qquad b = 1 + d.
$$

Тогда нижняя граница $95\%$ предиктивного интервала для доли будет вычисляться по такой формуле:

$$
\frac{a}{a + b} - 1.65 \cdot \sqrt{\frac{a \cdot b}{(a + b)^2 \cdot (a + b + 1)}}
$$

Если под новостью был всего один комментарий, у такой случайной величины будет высокая дисперсия. Это означаen, что штука, которую мы вычитаем из доли, окажется высокой. Левая граница интервала окажется маленькой и мы не поднимем комментарий в нашем топе наверх. Фактически мы делаем сортировку по квантилю уровня $0.05$ вместо среднего.

От вас требуется вбить эту формулу, сделать сортировку и вывести на экран топ позитивных новостей. Подробнее про то, откуда берётся эта формула можно почитать [в этой книге.](https://disk.yandex.ru/i/Ctd08bTwC9eI3g) Ищите 4 главу, страницу 140.

In [ ]:
u = df_vk['positive_comments'].astype(float)
d = (df_vk['negative_comments'] + df_vk['neutral_comments']).astype(float)
a = 1.0 + u
b = 1.0 + d

df_vk['pos_lower_bound'] = a / (a + b) - 1.65 * np.sqrt(a * b / ((a + b) ** 2 * (a + b + 1.0)))

mask = df_vk['total_comments_count'] >= 5
top_positive_news = df_vk[mask].nlargest(15, 'pos_lower_bound')
top_positive_news[['title', 'positive_comments', 'negative_comments', 'neutral_comments',
                   'total_comments_count', 'pos_lower_bound']]

Построили? Срочно пришлите свою любимую позитивную  новость в общий чат!!!

Топ негативных новостей строить не будем. Вокруг итак слишком много негатива 😻😻😻

__[0.2]__ Какие категории новостей оказались самыми позитивными? Придумайте способ найти такие категории и опишите его тут.

__Ответ:__

Применяем тот же байесовский трюк, что и для топа новостей, но агрегируем не на уровне поста, а на уровне категории:

1. Для каждой категории суммируем количество позитивных, негативных и нейтральных комментариев по всем новостям этой категории.
2. Считаем `u` = всего позитивных, `d` = всего нейтральных + негативных.
3. Считаем нижнюю границу 95%-го предиктивного интервала для доли позитивных:

$$
LB = \frac{a}{a+b} - 1.65 \sqrt{\frac{ab}{(a+b)^2 (a+b+1)}}, \quad a = 1+u, \; b = 1+d.
$$

4. Чтобы не вытаскивать наверх категории, у которых пара комментариев под одной новостью, отфильтровываем категории с малым числом комментариев (`>=50`) и малым числом новостей (`>=5`).
5. Сортируем по `pos_lower_bound` по убыванию.

Так маленькие, но «случайно позитивные» категории не пробьются в топ — а большие категории с устойчивой позитивной долей попадут.

In [ ]:
df_vk_exp = df_vk.explode('target_tags').dropna(subset=['target_tags'])
cat_stats = df_vk_exp.groupby('target_tags').agg(
    total_pos=('positive_comments', 'sum'),
    total_neg=('negative_comments', 'sum'),
    total_neu=('neutral_comments', 'sum'),
    n_news=('id', 'count'),
).astype(float)
cat_stats['total'] = cat_stats['total_pos'] + cat_stats['total_neg'] + cat_stats['total_neu']

cat_stats = cat_stats[(cat_stats['total'] >= 50) & (cat_stats['n_news'] >= 5)]

a = 1.0 + cat_stats['total_pos']
b = 1.0 + cat_stats['total_neg'] + cat_stats['total_neu']
cat_stats['pos_share'] = cat_stats['total_pos'] / cat_stats['total']
cat_stats['pos_lower_bound'] = a / (a + b) - 1.65 * np.sqrt(a * b / ((a + b) ** 2 * (a + b + 1.0)))

top_cats = cat_stats.nlargest(15, 'pos_lower_bound')
print('Топ-15 самых позитивных категорий по нижней границе доверительного интервала:')
print(top_cats[['n_news', 'total', 'pos_share', 'pos_lower_bound']].round(4))

__[0.2 бонусных]__ Проанализируйте, как температура комментария (вероятность того, что он негативный) зависит от длины трэда (число комментариев под новостью)? Значима ли эта взаимосвязь? Если вам для проверки этого хочется построить линейную регрессию, не сдерживайтесь.

In [ ]:
from sklearn.linear_model import LinearRegression
import scipy.stats as stats

per_post = df_comments.groupby('post_id').agg(
    thread_len=('text', 'size'),
    n_neg=('sentiment_verdict', lambda s: (s == 'negative').sum()),
).reset_index()
per_post['neg_prob'] = per_post['n_neg'] / per_post['thread_len']
per_post = per_post[per_post['thread_len'] >= 1]

corr, pval = stats.pearsonr(per_post['thread_len'], per_post['neg_prob'])
sp_corr, sp_pval = stats.spearmanr(per_post['thread_len'], per_post['neg_prob'])
print(f'Pearson:  r = {corr:.4f}, p = {pval:.3e}')
print(f'Spearman: r = {sp_corr:.4f}, p = {sp_pval:.3e}')

X = np.log1p(per_post[['thread_len']].values)
y = per_post['neg_prob'].values
lr = LinearRegression().fit(X, y)
print(f'\nLinear regression on log(1 + thread_len):')
print(f'  coef = {lr.coef_[0]:.4f}, intercept = {lr.intercept_:.4f}, R^2 = {lr.score(X, y):.4f}')

plt.figure(figsize=(10, 6))
plt.scatter(per_post['thread_len'], per_post['neg_prob'], alpha=0.2, s=6)
xs_raw = np.linspace(1, per_post['thread_len'].max(), 200)
xs_log = np.log1p(xs_raw).reshape(-1, 1)
plt.plot(xs_raw, lr.predict(xs_log), color='red', linewidth=2, label='LinReg')
plt.xscale('log')
plt.xlabel('Длина треда (число комментариев)')
plt.ylabel('Доля негативных комментариев')
plt.title('Зависимость негативности от длины обсуждения')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

if pval < 0.05:
    direction = 'положительная' if corr > 0 else 'отрицательная'
    print(f'\nВзаимосвязь статистически значима: {direction} корреляция между длиной треда и долей негатива.')
else:
    print('\nЗначимой линейной взаимосвязи не обнаружено.')